# 웹크롤링 (Lv1)

## 1. 전체 처리 흐름 (Workflow)

```
[1] HTTP 요청 및 HTML 가져오기 (requests)
               ▼
[2] HTML 파싱 및 DOM 트리 생성 (BeautifulSoup)
               ▼
[3] 데이터 추출 (상단 기본 정보 + 하단 상세 테이블)
               ▼
[4] 데이터 통합 및 Pandas 출력 (Series 변환)

```

## 2. 핵심 기능별 코드 설명

### [1] 대상 페이지 접속 및 준비

- **requests.get(URL, headers=headers)**: 봇(Bot) 차단을 우회하기 위해 브라우저 정보(User-Agent)를 헤더에 포함하여 웹 서버에 페이지 데이터를 요청합니다.
- **BeautifulSoup(response.text, "html.parser")**: 서버로부터 받은 HTML 텍스트를 파이썬이 분석할 수 있는 객체(DOM 트리)로 변환합니다.

### [2] 상단 핵심 영역 정보 추출

- **도서 제목**: `<div class="product_main">` 내부에 있는 첫 번째 `<h1>` 태그의 텍스트를 가져옵니다.
- **도서 요약 설명**: `id="product_description"`인 요소 바로 다음에 오는 `<p>` 태그를 `find_next_sibling("p")`로 찾아내어 텍스트를 추출합니다. (줄바꿈 공백 노드를 건너뛰는 안전한 방법)

### [3] 하단 제품 상세 테이블 추출

- `<table class="table table-striped">` 내부에 존재하는 모든 행(`<tr>`)을 반복문으로 돕니다.
- 각 행에서 항목명(`<th>`)과 데이터 값(`<td>`)을 1:1로 매핑하여 `product_info`라는 딕셔너리에 저장합니다. (예: UPC, 가격, 재고 등)

### [4] 데이터 병합 및 시각화

- 추출한 개별 변수(title, description)와 딕셔너리(product_info)의 데이터를 하나의 `book_details` 딕셔너리로 통합합니다.
- 데이터의 가독성을 높이기 위해 `pd.Series(book_details)`를 사용해 키-값 형태의 구조로 콘솔에 출력합니다.

---

## 주요 Key Point

- **find_next_sibling() 활용**: HTML 구조상 제목 태그와 설명 내용 태그가 나란히 배치되어 있을 때, 공백(텍스트 노드)을 무시하고 원하는 다음 태그를 정확히 집어내는 유용한 테크닉입니다.
- **product_info.get("Key") 사용**: 테이블에 특정 항목이 없을 경우 프로그램이 멈추는(KeyError) 현상을 방지하기 위해 `dict["key"]` 대신 안전한 `.get()` 메서드를 사용했습니다.


## 웹 크롤링 핵심 개념 및 라이브러리 가이드

### 1. 라이브러리 소개
#### requests (HTTP 통신 라이브러리)

- **역할**: 파이썬에서 웹 서버에 데이터를 요청(Request)하고 응답(Response)을 받는 역할을 합니다.
- **특징**: 웹 브라우저가 주소창에 URL을 입력하고 엔터를 누르는 과정을 코드로 구현해 줍니다. 이미지, HTML, JSON 등 다양한 형태의 웹 데이터를 가져올 수 있습니다.

#### BeautifulSoup (HTML 파싱 라이브러리)

- **역할**: 가져온 복잡한 HTML 텍스트를 파이썬이 이해하고 컴퓨터 구조처럼 탐색할 수 있는 객체(DOM 트리)로 변환합니다.
- **특징**: `find()`, `find_all()`, `find_next_sibling()` 같은 직관적인 메서드를 제공하여 HTML 태그나 클래스명, ID를 기준으로 원하는 데이터만 자석처럼 뽑아낼 수 있게 돕습니다.


---

### 2. 웹 크롤링에서 네트워크 이해가 중요한 이유

코드로 웹 페이지를 긁어오는 크롤링은 결국 **"웹 서버와의 통신"** 프로세스입니다. 따라서 기본적인 웹 네트워크 구조를 이해해야 효율적이고 안전한 크롤링이 가능합니다.

#### 클라이언트-서버 구조 (Request / Response)

- 크롤러는 '클라이언트(요청자)' 역할을 합니다. 서버에 "이 주소의 HTML을 주세요"라고 요청(GET)하면, 서버는 그에 따른 '상태 코드'와 함께 데이터를 돌려줍니다.
- 코드의 `if response.status_code == 200:` 부분처럼, 네트워크 상태를 확인하는 과정이 필수적입니다. (200은 성공, 404는 페이지 없음, 403/500은 접근 제한 및 서버 에러)

#### 차단 우회와 헤더(Header) 설정

- 웹 서버는 짧은 시간 안에 수많은 요청을 보내는 매크로 프로그램을 공격(DDoS)이나 자원 낭비로 판단하여 차단합니다.
- 코드에 포함된 `headers = {"User-Agent": "..."}`는 서버에게 "나는 프로그램이 아니라 실제 크롬 브라우저를 쓰는 사람입니다"라고 속이는 일종의 신분증 역할을 합니다. 이 네트워크 개념을 모르면 크롤러가 금방 차단당하게 됩니다.

#### 정적 데이터 vs 동적 데이터 구조 차이

- `requests`는 서버가 처음에 완성해서 보내주는 HTML(정적 데이터)만 가져올 수 있습니다.
- 최근 웹사이트들은 사용자가 스크롤을 내리거나 버튼을 누를 때 자바스크립트(JS)를 통해 뒤늦게 데이터를 불러오는 구조(동적 데이터)가 많습니다. 네트워크 탭을 통해 데이터가 오가는 흐름을 볼 줄 알아야 어떤 방식으로 크롤링할지(Selenium을 쓸지, API 주소를 직접 딸지) 결정할 수 있습니다.


### 웹 및 네트워크 기초 추천 키워드

* **MDN 웹 문서 (Mozilla Developer Network)**: 'HTTP 개요' 및 'HTTP 요청 메서드'를 검색하여 읽어보시면 클라이언트와 서버가 대화하는 기본 원리를 쉽게 이해할 수 있습니다.
* **크롬 개발자 도구 활용법**: 브라우저에서 `F12`를 눌러 [Network] 탭을 확인하는 방법을 유튜브나 블로그를 통해 숙지하시면, 웹 페이지 뒷단에서 어떤 데이터가 오가는지 눈으로 직접 확인할 수 있어 크롤링 실력이 비약적으로 상승합니다.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. 특정 도서 상세 페이지 URL 및 HTTP 요청 헤더 설정
URL = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("도서 상세 정보 수집 중...")
response = requests.get(URL, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    # -------------------------------------------------------------------------
    # [1] 상단 핵심 영역 정보 추출 (도서명, 가격, 별점 등)
    # -------------------------------------------------------------------------
    # 도서 제목 (main 레이아웃 내부의 <h1> 태그 추출)
    title = soup.find("div", class_="product_main").h1.text

    # 도서 요약 설명 (id가 product_description인 태그 다음(next_sibling) 나오는 <p> 태그 추출)
    # 태그 사이의 줄바꿈 노드를 건너뛰기 위해 find_next_sibling 사용
    desc_tag = soup.find("div", id="product_description")
    description = desc_tag.find_next_sibling("p").text if desc_tag else "No description available"

    # -------------------------------------------------------------------------
    # [2] 하단 제품 상세 테이블(Product Information) 정보 추출
    # -------------------------------------------------------------------------
    # 하단에 배치된 <table> 내부의 모든 행(<tr>)을 가져와 딕셔너리로 매핑합니다.
    info_table = soup.find("table", class_="table table-striped")
    product_info = {}

    if info_table:
        for row in info_table.find_all("tr"):
            header = row.find("th").text.strip() # 테이블 좌측 항목명 (예: UPC, Price (excl. tax))
            value = row.find("td").text.strip()  # 테이블 우측 실제 데이터 값
            product_info[header] = value

    # -------------------------------------------------------------------------
    # [3] 추출 데이터 병합 및 정리
    # -------------------------------------------------------------------------
    # 상단 기본 정보와 하단 테이블 정보를 하나의 딕셔너리로 통합합니다.
    book_details = {
        "Title": title,
        "Description": description,
        "UPC (고유코드)": product_info.get("UPC"),
        "Product Type": product_info.get("Product Type"),
        "Price (excl. tax)": product_info.get("Price (excl. tax)"),
        "Price (incl. tax)": product_info.get("Price (incl. tax)"),
        "Tax": product_info.get("Tax"),
        "Availability": product_info.get("Availability"),
        "Number of reviews": product_info.get("Number of reviews")
    }

    # 데이터 가독성을 위해 Pandas 시리즈 객체로 변환 후 출력
    s_book = pd.Series(book_details)
    print("\n==================== [도서 상세 정보 스크랩 완료] ====================")
    print(s_book)
    print("=======================================================================")

else:
    print(f"웹 페이지 연결 실패 (에러 코드: {response.status_code})")

도서 상세 정보 수집 중...

==================== [도서 상세 정보 스크랩 완료] ====================
Title                                             A Light in the Attic
Description          It's hard to imagine a world without A Light i...
UPC (고유코드)                                            a897fe39b1053632
Product Type                                                     Books
Price (excl. tax)                                              Â£51.77
Price (incl. tax)                                              Â£51.77
Tax                                                             Â£0.00
Availability                                   In stock (22 available)
Number of reviews                                                    0
dtype: object


# 웹크롤링 Lv(2)

- 다중 URL 순회 웹 크롤링 코드 분석 요약
  + 이 스크립트는 메인 화면에서 여러 도서의 상세 페이지 주소(URL)를 먼저 확보
  + 해당 주소들을 반복문으로 방문하여 데이터를 누적하고 Pandas 데이터프레임으로 최종 시각화하는 구조

## 1. 전체 처리 흐름 (Workflow)

```
[1단계: 메인 페이지] 20개 도서의 상세 URL 수집 및 절대 경로 변환
                         ▼
[2단계: 상세 페이지 순회] 0.5초 간격(time.sleep)으로 각 URL 접속 및 데이터 파싱
                         ▼
[3단계: 데이터 수집/정제] 인코딩 깨짐 수정 및 20개 도서 딕셔너리를 리스트에 축적
                         ▼
[4단계: 데이터프레임 변환] pd.DataFrame()을 통해 마스터 테이블 생성 및 출력

```

---

## 2. 핵심 기능 및 문법 설명

### 1단계: URL 수집 및 절대 경로 결합

- **soup.find_all("article", class_="product_pod")**: 메인 페이지에 배치된 20개의 도서 블록을 모두 찾아냅니다. 파이썬 예약어인 `class`와의 충돌을 막기 위해 `class_` 매개변수를 사용했습니다.
- **상대 경로를 절대 경로로 변환**: `book.h3.a["href"]`를 통해 추출한 주소는 상대 경로이므로, 앞 부분에 메인 주소(`BASE_URL`)를 더해 완성된 URL(`https://books.toscrape.com/catalogue/...`)을 만들어 리스트에 저장합니다.

### 2단계: 안전한 루프 제어 및 매너 크롤링

- **enumerate(detail_urls, 1)**: 현재 몇 번째 도서를 수집하고 있는지 번호(index)를 매겨 진행 상황을 시각적으로 파악할 수 있게 합니다.
- **time.sleep(0.5)**: 요청과 요청 사이에 0.5초의 휴식기를 줍니다. 서버에 가해지는 부하를 줄여 차단 가능성을 낮추고, 대상 서버의 자원을 보호하는 크롤링의 필수 매너이자 필수 테크닉입니다.

### 3단계: 인코딩 예외 처리 및 데이터 정제

- **.replace("Â", "")**: 일부 웹 페이지에서 통신 인코딩 문제로 인해 파운드 기호(£) 앞에 깨진 문자(`Â`)가 붙는 현상이 발생합니다. 이를 `.replace()` 메서드로 깔끔하게 지워 데이터의 순수성을 높였습니다.

### 4단계: Pandas DataFrame 마스터 테이블 생성

- **pd.DataFrame(all_book_details)**: 딕셔너리들이 담긴 리스트를 2차원 데이터프레임(표) 구조로 즉시 변환합니다.
- **df_final_books.head()**: 수집된 데이터가 너무 길어 콘솔창을 덮는 것을 방지하기 위해 상위 5개의 행만 샘플로 출력합니다.
- **df_final_books.shape**: 최종 결과물의 형상(행의 개수, 열의 개수)을 확인하여 데이터가 누락 없이 20행으로 잘 들어왔는지 검증합니다.

---

## 주요 Key Point

- **2단계 구조의 크롤링**: 목록에서 링크를 따고 상세 페이지로 들어가는 방식은 실제 현업 크롤링에서 가장 많이 쓰이는 정석적인 패턴입니다.
- **서버 차단 방지(Anti-Blocking)**: `User-Agent` 설정과 `time.sleep()`의 조합은 봇 차단 시스템을 우회하고 안정적인 데이터 수집을 보장하는 가장 기본적인 방어선입니다.


In [ ]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 메인 베이스 URL 및 1페이지 주소 설정
BASE_URL = "https://books.toscrape.com/"
START_URL = "https://books.toscrape.com/index.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("1단계: 1페이지에 있는 20개 도서의 상세 페이지 링크 수집 중...")
response = requests.get(START_URL, headers=headers)

if response.status_code != 200:
    print(f"메인 페이지 접속 실패 (에러 코드: {response.status_code})")
    exit()

soup = BeautifulSoup(response.text, "html.parser")

# [수정 완료] class 뒤에 언더바(_)를 붙여 파이썬 예약어와의 문법 충돌을 방지합니다.
books = soup.find_all("article", class_="product_pod")

# 20개 도서의 상대 경로 주소를 절대 경로 URL로 변환하여 리스트에 축적
detail_urls = []
for book in books:
    # 예: "catalogue/a-light-in-the-attic_1000/index.html"
    relative_url = book.h3.a["href"]
    # 절대 경로로 결합 (https://books.toscrape.com/catalogue/...)
    full_url = BASE_URL + relative_url
    detail_urls.append(full_url)

print(f"▶ 총 {len(detail_urls)}개의 도서 상세 URL 수집 완료.")
print("-" * 60)

# -------------------------------------------------------------------------
# 2단계: 수집된 20개 URL을 순회하며 상세 정보 스크랩 실행
# -------------------------------------------------------------------------
all_book_details = []

for idx, url in enumerate(detail_urls, 1):
    print(f"[{idx}/20] 수집 중: {url.split('/')[-2]}")

    # 웹 서버 차단 방지 및 매너 크롤링을 위한 0.5초 대기
    time.sleep(0.5)

    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print(f"└ [경고] 해당 페이지 접속 실패 스킵합니다. (코드: {res.status_code})")
        continue

    detail_soup = BeautifulSoup(res.text, "html.parser")

    # [상단 영역] 도서명 및 설명 추출
    # 여기도 class_ 문법으로 안전하게 적용되어 있습니다.
    title = detail_soup.find("div", class_="product_main").h1.text

    desc_tag = detail_soup.find("div", id="product_description")
    description = desc_tag.find_next_sibling("p").text if desc_tag else "No description"

    # [하단 영역] 제품 스펙 테이블 파싱
    info_table = detail_soup.find("table", class_="table table-striped")
    product_info = {}
    if info_table:
        for row in info_table.find_all("tr"):
            header = row.find("th").text.strip()
            value = row.find("td").text.strip()
            product_info[header] = value

    # 데이터 구조 통합 및 텍스트 정제 (Â£ 기호 파운드 기호 '£'로 치환)
    all_book_details.append({
        "Title": title,
        "UPC": product_info.get("UPC"),
        "Product Type": product_info.get("Product Type"),
        "Price (excl. tax)": product_info.get("Price (excl. tax)").replace("Â", ""),
        "Price (incl. tax)": product_info.get("Price (incl. tax)").replace("Â", ""),
        "Tax": product_info.get("Tax").replace("Â", ""),
        "Availability": product_info.get("Availability"),
        "Number of reviews": product_info.get("Number of reviews"),
        "Description": description
    })

# -------------------------------------------------------------------------
# 3단계: 최종 Pandas DataFrame 생성 및 결과 확인
# -------------------------------------------------------------------------
df_final_books = pd.DataFrame(all_book_details)

print("\n==================== [최종 20개 도서 상세 정보 마스터 DataFrame] ====================")
# 데이터프레임의 상단 5개 행 미리보기 출력
print(df_final_books.head())
print(f"\n▶ 데이터프레임 최종 형상 (행, 열): {df_final_books.shape}")
print("====================================================================================")

1단계: 1페이지에 있는 20개 도서의 상세 페이지 링크 수집 중...
▶ 총 20개의 도서 상세 URL 수집 완료.
------------------------------------------------------------
[1/20] 수집 중: a-light-in-the-attic_1000
[2/20] 수집 중: tipping-the-velvet_999
[3/20] 수집 중: soumission_998
[4/20] 수집 중: sharp-objects_997
[5/20] 수집 중: sapiens-a-brief-history-of-humankind_996
[6/20] 수집 중: the-requiem-red_995
[7/20] 수집 중: the-dirty-little-secrets-of-getting-your-dream-job_994
[8/20] 수집 중: the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993
[9/20] 수집 중: the-boys-in-the-boat-nine-americans-and-their-epic-quest-for-gold-at-the-1936-berlin-olympics_992
[10/20] 수집 중: the-black-maria_991
[11/20] 수집 중: starving-hearts-triangular-trade-trilogy-1_990
[12/20] 수집 중: shakespeares-sonnets_989
[13/20] 수집 중: set-me-free_988
[14/20] 수집 중: scott-pilgrims-precious-little-life-scott-pilgrim-1_987
[15/20] 수집 중: rip-it-up-and-start-again_986
[16/20] 수집 중: our-band-could-be-your-life-scenes-from-the-american-indie-underground-1981-19

# 웹크롤링 Lv(3)

- 전체 페이지 자동 순회(Pagination) 웹 크롤링 코드 분석 요약
  + 이 스크립트는 하단 네비게이션의 'Next' 버튼 존재 여부를 검사하여 다음 페이지가 없을 때까지 `while` 루프를 동적으로 구동
  + 웹 프레임워크의 구조적 특징(상대 경로)을 처리하여 전체 데이터를 마스터 CSV 파일로 저장하는 흐름을 가집니다.

## 1. 전체 처리 흐름 (Workflow)

```
[1단계: while 루프 진입] current_url이 존재하면 계속해서 페이지 HTML 요청
                              ▼
[2단계: 현재 페이지 스크랩] 내부의 20개 도서 상세 URL을 절대 경로로 보정 후 데이터 적재
                              ▼
[3단계: 다음 페이지 추적] <li class="next"> 태그 검색
                          ├── 존재함 -> current_url 갱신, page_count +1 후 루프 재개
                          └── 존재 안 함 -> current_url = None 변경 후 루프 탈출
                              ▼
[4단계: 데이터셋 빌드] pd.DataFrame 생성, 소요 시간 계산 및 'to_csv()'를 통한 최종 저장

```

---

## 2. 핵심 메커니즘 및 문법 설명

### while 주소 자동 갱신 (Dynamic Loop)

- `while current_url:` 구조를 사용하여 조건문이 참(`None`이 아님)인 동안 무한히 반복됩니다.
- 마지막 페이지에 도달하여 'Next' 버튼을 찾지 못하면 `current_url = None` 처리가 되어 자연스럽게 루프를 탈출하도록 설계되었습니다. 이는 고정된 페이지 수를 모를 때 사용하는 가장 표준적인 패턴입니다.

### 상대 경로 정제 및 절대 경로 구축

- 목록 페이지에서 추출한 주소에 포함된 `../` 기호를 `.replace("../", "")`로 제거한 뒤 `BASE_URL + "catalogue/"`와 결합합니다.
- 크롤러가 페이지를 이동하더라도 항상 올바른 상세 페이지 주소(`https://books.toscrape.com/catalogue/...`)를 가리키도록 도메인 맥락을 맞추는 정밀한 예외 처리입니다.

### time 라이브러리를 활용한 성능 및 매너 관리

- **time.time()**: 프로세스 시작 시점과 종료 시점의 타임스탬프를 기록하여 전체 소요 시간 및 페이지당 평균 처리 속도를 연산합니다.
- **time.sleep(0.2) 및 time.sleep(0.5)**: 도서 상세 페이지 내부 진입 시와 페이지를 넘겨갈 때 각각 휴식 시간을 차등 부여하여, 연속적인 대량의 HTTP 요청으로 인해 발생할 수 있는 IP 차단 및 서버 과부하를 방지합니다.

### 데이터 파일 영구 저장

- **df_all_books.to_csv("...", index=False, encoding="utf-8-sig")**: 수집된 메모리 상의 데이터프레임을 물리적인 CSV 파일로 내보냅니다.
- **encoding="utf-8-sig"**: 엑셀(Excel) 프로그램에서 CSV 파일을 열었을 때 한글이나 파운드(£) 같은 특수 문자가 깨지는 현상을 방지하는 윈도우 환경 최적화 옵션입니다.

---

## 주요 Key Point

- **자동화 크롤러의 완성**: 수작업으로 페이지 번호를 바꿀 필요 없이 웹사이트의 UI 구성 요소를 추적하여 데이터의 시작과 끝을 스스로 판단하는 완성도 높은 크롤러입니다.
- **수집 결과 검증(Shape)**: `df_all_books.shape`를 통해 행과 열의 길이를 파악하여 총 수집 데이터의 누락 여부를 직관적으로 검증할 수 있습니다.


In [ ]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 메인 주소 및 catalogue 도메인 경로 설정
BASE_URL = "https://books.toscrape.com/"
# 1페이지 링크 (상세 페이지 주소 변환 규칙을 맞추기 위해 catalogue 경로로 진입)
current_url = "https://books.toscrape.com/catalogue/page-1.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

all_book_details = []
page_count = 1

print("==================== [전체 페이지 자동 순회 크롤링 시작] ====================")

# [시간 측정 시작] 크롤링 전체 프로세스의 시작 시간을 기록합니다.
start_time = time.time()

# [무한 루프 진입] 다음 페이지('Next') 링크가 존재하는 동안 계속해서 실행됩니다.
while current_url:
    # 각 페이지별 소요 시간 측정을 위한 로컬 타이머 시작
    page_start_time = time.time()
    print(f"\n[현재 {page_count}페이지] 주소 요청 중: {current_url}")

    response = requests.get(current_url, headers=headers)
    if response.status_code != 200:
        print(f"└ [에러] 해당 페이지 접속 실패로 루프를 강제 종료합니다. (코드: {response.status_code})")
        break

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    # -------------------------------------------------------------------------
    # 1단계: 현재 페이지 내부의 도서 상세 URL 수집 및 스크랩
    # -------------------------------------------------------------------------
    for idx, book in enumerate(books, 1):
        # 현재 위치가 catalogue 폴더 내부이므로 상대 경로를 직접 결합
        # 예: "a-light-in-the-attic_1000/index.html" -> 베이스와 붙여 완벽한 절대경로 구축
        relative_url = book.h3.a["href"]
        detail_url = BASE_URL + "catalogue/" + relative_url.replace("../", "")

        # 서버 과부하 방지용 매너 인터벌 (0.2초)
        time.sleep(0.2)

        res = requests.get(detail_url, headers=headers)
        if res.status_code != 200:
            continue

        detail_soup = BeautifulSoup(res.text, "html.parser")

        # 상단 영역 파싱
        title = detail_soup.find("div", class_="product_main").h1.text
        desc_tag = detail_soup.find("div", id="product_description")
        description = desc_tag.find_next_sibling("p").text if desc_tag else "No description"

        # 하단 스펙 테이블 파싱
        info_table = detail_soup.find("table", class_="table table-striped")
        product_info = {}
        if info_table:
            for row in info_table.find_all("tr"):
                header = row.find("th").text.strip()
                value = row.find("td").text.strip()
                product_info[header] = value

        # 데이터 클렌징 후 적재
        all_book_details.append({
            "Page_Num": page_count,
            "Title": title,
            "UPC": product_info.get("UPC"),
            "Product Type": product_info.get("Product Type"),
            "Price (excl. tax)": product_info.get("Price (excl. tax)").replace("Â", ""),
            "Price (incl. tax)": product_info.get("Price (incl. tax)").replace("Â", ""),
            "Tax": product_info.get("Tax").replace("Â", ""),
            "Availability": product_info.get("Availability"),
            "Number of reviews": product_info.get("Number of reviews"),
            "Description": description
        })

    page_end_time = time.time()
    print(f"└ {page_count}페이지 수집 완료 (소요 시간: {page_end_time - page_start_time:.2f}초 / 누적 수집 데이터 수: {len(all_book_details)}개)")

    # -------------------------------------------------------------------------
    # 2단계: [핵심 메커니즘] 'Next' 버튼을 찾아 다음 페이지 URL로 업데이트
    # -------------------------------------------------------------------------
    # 하단 네비게이션바의 <li class="next"> 태그 내부의 <a> 태그를 탐색합니다.
    next_btn = soup.find("li", class_="next")

    if next_btn and next_btn.a:
        # 다음 페이지 상대 경로 추출 (예: "page-2.html")
        next_page_url = next_btn.a["href"]
        # 다음 순회할 절대 경로 완성
        current_url = BASE_URL + "catalogue/" + next_page_url
        page_count += 1

        # 페이지 전환 시 안정적인 세션 유지를 위해 0.5초 휴식
        time.sleep(0.5)
    else:
        # 더 이상 'Next' 버튼이 존재하지 않는 경우 (마지막 페이지에 도달함)
        print("\n▶ [안내] 다음 페이지('Next') 버튼을 찾을 수 없습니다. 최종 페이지에 도달했습니다.")
        current_url = None # while 루프 조건문을 거짓(False)으로 만들어 루프 탈출

# [시간 측정 종료] 전체 크롤링 프로세스가 끝난 시점의 연산입니다.
end_time = time.time()
total_execution_time = end_time - start_time

# -------------------------------------------------------------------------
# 3단계: 최종 데이터프레임 빌드 및 파일 저장
# -------------------------------------------------------------------------
df_all_books = pd.DataFrame(all_book_details)

print("\n==================== [전체 도서 상세 정보 마스터 데이터프레임 빌드 완료] ====================")
print(f"▶ 총 수집 페이지 수 : {page_count} 페이지")
print(f"▶ 전체 소요 시간    : {total_execution_time:.2f} 초 (약 {total_execution_time/60:.1f} 분)")
print(f"▶ 페이지당 평균 시간 : {total_execution_time / page_count:.2f} 초/페이지")
print(f"▶ 데이터프레임 최종 형상 (행, 열): {df_all_books.shape}")
print("-" * 92)
print(df_all_books.head()) # 상단 5개 샘플 데이터 출력
print("==========================================================================================")

# 최종 전체 데이터셋 파일 저장 (구글 코랩 좌측 파일 탭에서 다운로드 가능)
df_all_books.to_csv("all_books_master.csv", index=False, encoding="utf-8-sig")
print("▶ 'all_books_master.csv' 저장 완료.")

==================== [전체 페이지 자동 순회 크롤링 시작] ====================

[현재 1페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-1.html
└ 1페이지 수집 완료 (소요 시간: 4.87초 / 누적 수집 데이터 수: 20개)

[현재 2페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-2.html
└ 2페이지 수집 완료 (소요 시간: 4.64초 / 누적 수집 데이터 수: 40개)

[현재 3페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-3.html
└ 3페이지 수집 완료 (소요 시간: 4.55초 / 누적 수집 데이터 수: 60개)

[현재 4페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-4.html
└ 4페이지 수집 완료 (소요 시간: 4.83초 / 누적 수집 데이터 수: 80개)

[현재 5페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-5.html
└ 5페이지 수집 완료 (소요 시간: 4.62초 / 누적 수집 데이터 수: 100개)

[현재 6페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-6.html
└ 6페이지 수집 완료 (소요 시간: 4.84초 / 누적 수집 데이터 수: 120개)

[현재 7페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-7.html
└ 7페이지 수집 완료 (소요 시간: 5.14초 / 누적 수집 데이터 수: 140개)

[현재 8페이지] 주소 요청 중: https://books.toscrape.com/catalogue/page-8.html
└ 8페이지 수집 완료 (소요 시간: 4.55초 / 누적 수집 데이터 수: 160개)

[현재

# 웹크롤링 Lv(4)

- **멀티스레딩(Multi-threading)** 기술을 도입한 **고급 단계(Lv4)의 병렬 크롤링 스크립트**
  + 이 스크립트는 네트워크 요청 대기 시간(I/O Bound)을 최소화하기 위해 `concurrent.futures` 모듈의 `ThreadPoolExecutor`를 활용
  + 1단계에서 URL 목록을 싱글스레드로 빠르게 확보
  + 2단계에서 10개의 스레드가 동시에 상세 페이지를 긁어오는 구조를 취하고 있습니다.

## 1. 전체 처리 흐름 (Workflow)

```
[Phase 1] 싱글스레드 순회 -> 전체 페이지를 돌며 도서 상세 URL 리스트(detail_urls) 고속 수집
                                 ▼
[Phase 2] 병렬 작업 정의 -> 단일 URL을 파싱하여 딕셔너리로 반환하는 독립 함수(scrape_book_detail) 선언
                                 ▼
[Phase 3] 멀티스레드 실행 -> ThreadPoolExecutor(max_workers=10)를 통해 10개씩 동시 HTTP 요청 수행
                                 ▼
[Phase 4] 실시간 취합 -> as_completed()로 먼저 끝난 작업부터 순서대로 결과 리스트에 적재
                                 ▼
[Phase 5] 최종 마스터 빌드 -> Pandas 데이터프레임 변환 및 엑셀 깨짐 방지(utf-8-sig) 반영 후 CSV 저장

```

---

## 2. 핵심 메커니즘 및 문법 설명

### 작업 단위의 함수화 (Decoupling)

- **scrape_book_detail(url)**: 멀티스레딩을 구현하려면 각 스레드가 독립적으로 실행할 '작업 단위'가 필요합니다. 하나의 URL을 받아 성공 시 딕셔너리를, 실패 시 `None`을 반환하는 함수를 독립적으로 분리했습니다.
- **try-except 예외 처리**: 병렬 처리 중 특정 페이지에서 네트워크 유실이나 타임아웃(`timeout=5`) 에러가 발생하더라도, 전체 크롤링 파이프라인이 멈추지 않고 다른 스레드의 작업이 계속 유지되도록 안전장치를 마련했습니다.

### ThreadPoolExecutor를 통한 병렬화 (I/O Bound 최적화)

- 웹 크롤링은 컴퓨터 연산보다 '서버의 응답을 기다리는 시간(I/O 대기)'이 대부분을 차지합니다.
- **max_workers=10**: 10개의 스레드(일꾼)를 동시에 가동하여, 1번 일꾼이 서버 응답을 기다리는 동안 2~10번 일꾼이 동시에 다른 주소로 요청을 보냅니다. 이를 통해 싱글스레드 대비 이론상 최대 10배에 가까운 속도 향상을 이끌어냅니다.

### Future 객체와 as_completed() 실시간 취합

- **executor.submit()**: 작업을 스레드 풀에 예약하면, 즉시 결과가 나오는 것이 아니라 미래에 완료될 작업의 약속 증서인 `Future` 객체를 반환합니다.
- **as_completed(futures)**: 1번부터 10번까지의 순서와 상관없이, **가장 먼저 통신이 끝난 스레드의 결과물부터 실시간으로 꺼내어** 리스트에 추가합니다. 효율적인 자원 소모와 실시간 진행 상황 브리핑(`idx % 50 == 0`)을 가능하게 만드는 핵심 매커니즘입니다.

---

## 주요 Key Point

- **속도와 매너의 트레이드오프(Trade-off)**: 멀티스레딩은 대량의 데이터를 수분 내에 긁어올 수 있는 강력한 무기이지만, 상대방 서버에 짧은 시간 동안 많은 트래픽을 유발합니다. 따라서 Target 서버의 보안 정책에 따라 `MAX_WORKERS` 수를 적절히 조율해야 하며, 무차별적인 요청은 IP 차단이나 디도스(DDoS) 공격으로 오인받을 수 있으므로 주의가 필요합니다.
- **비동기 적재와 순서 보장**: `as_completed()`를 사용했기 때문에 최종 데이터프레임에 쌓이는 도서의 순서는 웹사이트에 등록된 순서가 아니라 '먼저 응답이 온 순서'로 뒤섞이게 됩니다. 순서가 중요하다면 수집된 데이터 내의 고유 키(예: UPC)를 기준으로 추후 정렬해야 합니다.

## 동기 vs 비동기 예
- 동기 (Synchronous) : `순서대로 줄 서서 기다리기`
  + 동기는 요청을 보낸 뒤, 그 요청의 결과(응답)가 올 때까지 아무것도 하지 않고 대기하는 방식
  + 모든 일이 직렬(순서대로) 구조로 진행됩니다.
  + 예시 : 마트 계산대에서 줄서서 기다리기
- 비동기 (Asynchronous) : "진동벨 받고 내 할 일 하기"
  + 비동기는 요청을 보내놓고, 그 결과가 나오든 말든 기단 부를 다른 일(다음 작업)을 즉시 진행하는 방식
  + 나중에 요청한 결과가 완료되면 그때 알림을 받아 처리합니다.
  + 예시 : 카페 진동벨
    - 대기할 필요 없이, 진동벨이 울리면 커피 받으러 가기
    - 계산대에서는 다음 손님들의 주문을 계속 받을 수 있음

### 동기/비동기 방식이 사용되어야 하는 곳
- 동기 프로그래밍이 필요한 곳 : 순서와 정확성이 우선시 되는 곳
  + 금융 시스템, 사용자 인증 권한 등
- 비동기 프로그래밍이 필요한 곳 : 멈춤(렉) 현상을 방지하고, 대기 시간을 효율적으로 쓰고 싶을 때
  + 대규모 서버의 네트워크 요청 처리, 스마트폰 앱 및 웹 UI (화면 멈춤 방지), 대용량 파일 업로드 등

In [ ]:
#import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# 공통 베이스 주소 및 헤더 설정
BASE_URL = "https://books.toscrape.com/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# =========================================================================
# [Phase 1] 동적 네비게이션 순회: 전체 도서의 상세 URL 고속 수집 (싱글스레드)
# =========================================================================
print("==================== [1단계: 전체 도서 상세 URL 고속 수집 시작] ====================")
current_url = START_URL
detail_urls = []
page_count = 1

start_time = time.time()

while current_url:
    response = requests.get(current_url, headers=HEADERS)
    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    # 현재 페이지의 도서 상대 경로를 절대 경로로 변환하여 수집
    for book in books:
        relative_url = book.h3.a["href"]
        detail_url = BASE_URL + "catalogue/" + relative_url.replace("../", "")
        detail_urls.append(detail_url)

    # 'Next' 버튼 추적을 통한 동적 페이지네이션
    next_btn = soup.find("li", class_="next")
    if next_btn and next_btn.a:
        current_url = BASE_URL + "catalogue/" + next_btn.a["href"]
        page_count += 1
    else:
        current_url = None # 마지막 페이지 도달 시 루프 종료

print(f"▶ [완료] 총 {page_count}개 페이지에서 {len(detail_urls)}개의 도서 상세 URL 수집 완료.")
print("-" * 80)


# =========================================================================
# [Phase 2] 멀티스레드 기반 도서 상세 정보 병렬 스크랩 함수 정의
# =========================================================================
def scrape_book_detail(url):
    """단일 도서 상세 페이지를 스크랩하여 딕셔너리로 반환하는 작업 단위 함수"""
    try:
        # 단일 요청 실패 시 전체 파이프라인이 멈추지 않도록 예외 처리(try-except) 필수
        res = requests.get(url, headers=HEADERS, timeout=5) # 5초 타임아웃 설정
        if res.status_code != 200:
            return None

        detail_soup = BeautifulSoup(res.text, "html.parser")

        # 기본 정보 파싱
        title = detail_soup.find("div", class_="product_main").h1.text
        desc_tag = detail_soup.find("div", id="product_description")
        description = desc_tag.find_next_sibling("p").text if desc_tag else "No description"

        # 기술 스펙 테이블 파싱
        info_table = detail_soup.find("table", class_="table table-striped")
        product_info = {}
        if info_table:
            for row in info_table.find_all("tr"):
                header = row.find("th").text.strip()
                value = row.find("td").text.strip()
                product_info[header] = value

        # 데이터 정제 후 반환
        return {
            "Title": title,
            "UPC": product_info.get("UPC"),
            "Product Type": product_info.get("Product Type"),
            "Price (excl. tax)": product_info.get("Price (excl. tax)", "").replace("Â", ""),
            "Price (incl. tax)": product_info.get("Price (incl. tax)", "").replace("Â", ""),
            "Tax": product_info.get("Tax", "").replace("Â", ""),
            "Availability": product_info.get("Availability"),
            "Number of reviews": product_info.get("Number of reviews"),
            "Description": description
        }
    except Exception as e:
        # 네트워크 유실 등으로 인한 에러 발생 시 None 반환 후 패스
        return None

# =========================================================================
# [Phase 3] ThreadPoolExecutor를 활용한 멀티스레드 병렬 실행
# =========================================================================
print("==================== [2단계: 멀티스레드 기반 고속 병렬 스크랩 시작] ====================")
all_book_details = []

# max_workers=10: 동시에 10개의 스레드(일꾼)를 띄워 서버에 10개씩 동시 요청을 날립니다.
# 대상 사이트 사양과 차단 정책에 따라 5~15 사이를 조율하는 것이 안전합니다.
MAX_WORKERS = 10

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 수집한 URL 리스트를 스레드 풀에 매핑하여 비동기 작업(Future 객체)으로 예약
    futures = {executor.submit(scrape_book_detail, url): url for url in detail_urls}

    # 작업이 먼저 완료되는 순서대로 실시간 취합
    for idx, future in enumerate(as_completed(futures), 1):
        result = future.result()
        if result:
            all_book_details.append(result)

        # 50개 단위로 진행 상황 브리핑
        if idx % 50 == 0 or idx == len(detail_urls):
            print(f"▶ 병렬 처리 진행률: [{idx}/{len(detail_urls)}] 완료 (누적 성공: {len(all_book_details)}개)")

# -------------------------------------------------------------------------
# 4. 최종 마스터 데이터프레임 빌드 및 벤치마크 결과 출력
# -------------------------------------------------------------------------
df_all_books = pd.DataFrame(all_book_details)
end_time = time.time()

print("\n==================== [고속 멀티스레드 크롤링 완결 리포트] ====================")
print(f"▶ 총 소요 시간: {end_time - start_time:.2f} 초")
print(f"▶ 최종 데이터프레임 형상 (행, 열): {df_all_books.shape}")
print("-" * 80)
print(df_all_books.head(3))
print("============================================================================")

# CSV 마스터 파일 최종 저장
df_all_books.to_csv("all_books_multithread_master.csv", index=False, encoding="utf-8-sig")

==================== [1단계: 전체 도서 상세 URL 고속 수집 시작] ====================
▶ [완료] 총 50개 페이지에서 1000개의 도서 상세 URL 수집 완료.
--------------------------------------------------------------------------------
==================== [2단계: 멀티스레드 기반 고속 병렬 스크랩 시작] ====================
▶ 병렬 처리 진행률: [50/1000] 완료 (누적 성공: 50개)
▶ 병렬 처리 진행률: [100/1000] 완료 (누적 성공: 100개)
▶ 병렬 처리 진행률: [150/1000] 완료 (누적 성공: 150개)
▶ 병렬 처리 진행률: [200/1000] 완료 (누적 성공: 200개)
▶ 병렬 처리 진행률: [250/1000] 완료 (누적 성공: 250개)
▶ 병렬 처리 진행률: [300/1000] 완료 (누적 성공: 300개)
▶ 병렬 처리 진행률: [350/1000] 완료 (누적 성공: 350개)
▶ 병렬 처리 진행률: [400/1000] 완료 (누적 성공: 400개)
▶ 병렬 처리 진행률: [450/1000] 완료 (누적 성공: 450개)
▶ 병렬 처리 진행률: [500/1000] 완료 (누적 성공: 500개)
▶ 병렬 처리 진행률: [550/1000] 완료 (누적 성공: 550개)
▶ 병렬 처리 진행률: [600/1000] 완료 (누적 성공: 600개)
▶ 병렬 처리 진행률: [650/1000] 완료 (누적 성공: 650개)
▶ 병렬 처리 진행률: [700/1000] 완료 (누적 성공: 700개)
▶ 병렬 처리 진행률: [750/1000] 완료 (누적 성공: 750개)
▶ 병렬 처리 진행률: [800/1000] 완료 (누적 성공: 800개)
▶ 병렬 처리 진행률: [850/1000] 완료 (누적 성공: 850개)
▶ 병렬 처리 진행률: [900/1000] 완료 (누적 성공: 900개

# API 활용한 크롤링